# Trajectory Optimization: Manual vs. DSPy Compilation
When agents fail at edge cases, zero-shot prompt engineering is rarely enough. The State-of-the-Art approach is providing **Gold Standard Trajectories** (Few-Shot examples). 

This notebook demonstrates how to do this manually using `langchain`, and how to automate it algorithmically using the industry standard `dspy` SDK.

**Dependencies required:** `pip install dspy-ai langchain`


## 1. Manual Few-Shot Trajectories (LangChain)
You can manually inject perfect execution traces into the system prompt to anchor the agent's behavior.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. Define the Gold Standard Trajectories (The "Few-Shot" examples)
examples = [
    {
        "input": "Cancel order 999",
        "output": "Thought: I need to check if order 999 is shipped.\nAction: get_order_status('999')\nObservation: Shipped\nThought: The order is shipped, cancellation is blocked.\nFinalAnswer: I'm sorry, but shipped orders cannot be canceled."
    },
    {
        "input": "Cancel order 111",
        "output": "Thought: I need to check if order 111 is shipped.\nAction: get_order_status('111')\nObservation: Processing\nThought: The order is still processing, I can cancel it.\nAction: execute_cancellation('111')\nObservation: Success\nFinalAnswer: Order 111 has been successfully canceled."
    }
]

# 2. Format them into a LangChain Prompt Template
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an order management agent. Follow the strict logic demonstrated below."),
        few_shot_prompt,
        ("human", "{user_input}"),
    ]
)

# Render the prompt
rendered = final_prompt.format(user_input="Cancel order 555")
print("📜 MANUALLY CONSTRUCTED PROMPT:\n")
print(rendered)


📜 MANUALLY CONSTRUCTED PROMPT:

System: You are an order management agent. Follow the strict logic demonstrated below.
Human: Cancel order 999
AI: Thought: I need to check if order 999 is shipped.
Action: get_order_status('999')
Observation: Shipped
Thought: The order is shipped, cancellation is blocked.
FinalAnswer: I'm sorry, but shipped orders cannot be canceled.
Human: Cancel order 111
AI: Thought: I need to check if order 111 is shipped.
Action: get_order_status('111')
Observation: Processing
Thought: The order is still processing, I can cancel it.
Action: execute_cancellation('111')
Observation: Success
FinalAnswer: Order 111 has been successfully canceled.
Human: Cancel order 555


## 2. Algorithmic Compilation (DSPy)
Manual examples are brittle. If you change a tool name, you have to rewrite your prompt. 
**DSPy** allows you to declare *what* you want (the Signature), and it automatically compiles the prompt by simulating the agent and extracting successful traces.


In [ ]:
import dspy

# In production, you would configure your LLM here:
# lm = dspy.OpenAI(model='gpt-4o-mini')
# dspy.settings.configure(lm=lm)

# 1. Define the Signature (Inputs and Outputs)
class OrderAgentSignature(dspy.Signature):
    """Determines if an order can be canceled and acts appropriately."""
    
    order_id = dspy.InputField(desc="The ID of the order.")
    order_status = dspy.InputField(desc="The current status of the order from the database.")
    
    agent_reasoning = dspy.OutputField(desc="Step-by-step logic determining if cancellation is allowed.")
    final_response = dspy.OutputField(desc="The final message to the user.")

# 2. Define the DSPy Module (The "Agent")
class OrderAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        # ChainOfThought automatically adds reasoning steps to the signature
        self.prog = dspy.ChainOfThought(OrderAgentSignature)
        
    def forward(self, order_id, order_status):
        return self.prog(order_id=order_id, order_status=order_status)

# 3. Define the Dataset (Inputs and Expected Outputs)
# Notice we don't write the "Thought" or "Action" manually! We just define the goal.
trainset = [
    dspy.Example(order_id="999", order_status="Shipped", final_response="I'm sorry, shipped orders cannot be canceled.").with_inputs("order_id", "order_status"),
    dspy.Example(order_id="111", order_status="Processing", final_response="Order 111 canceled successfully.").with_inputs("order_id", "order_status"),
    dspy.Example(order_id="222", order_status="Delivered", final_response="I'm sorry, delivered orders cannot be canceled.").with_inputs("order_id", "order_status")
]

# 4. The Teleprompter (The Compiler)
# BootstrapFewShot simulates the agent, scores it, and extracts the optimal trajectories.
from dspy.teleprompt import BootstrapFewShot

def validation_metric(example, pred, trace=None):
    # If the agent output matches the expected final response, it's a successful trajectory
    return example.final_response.lower() in pred.final_response.lower()

teleprompter = BootstrapFewShot(metric=validation_metric, max_bootstrapped_demos=2)

print("⚙️ [DSPy] Compiling the Agent...")
print("   - Simulating 3 runs...")
print("   - Extracting successful traces...")
print("   - Injecting traces as Few-Shot examples...")

# In a real environment, this line actually runs the LLM and compiles the optimized prompt:
# compiled_agent = teleprompter.compile(OrderAgent(), trainset=trainset)

print("✅ [DSPy] Agent compiled! You can now deploy `compiled_agent` to production.")
print("   If you switch from GPT-4o to Llama-3, you just rerun compile() and DSPy writes the new optimal prompt for you!")


⚙️ [DSPy] Compiling the Agent...
   - Simulating 3 runs...
   - Extracting successful traces...
   - Injecting traces as Few-Shot examples...
✅ [DSPy] Agent compiled! You can now deploy `compiled_agent` to production.
   If you switch from GPT-4o to Llama-3, you just rerun compile() and DSPy writes the new optimal prompt for you!
